In [49]:
import math
import sys
from pathlib import Path

_here = Path().resolve()
for _p in [_here, *_here.parents]:
    _src = _p / "src"
    if (_src / "qudits_on_qubits" / "__init__.py").is_file():
        repo_root = _p
        if str(_src) not in sys.path:
            sys.path.insert(0, str(_src))
        break
else:
    raise ImportError(
        "qudits_on_qubits repo root not found; run the notebook from notebooks/ or the repo root"
    )

from qiskit.quantum_info import Statevector, Operator, partial_trace, SparsePauliOp
from qiskit import qpy, QuantumCircuit
import numpy as np
from qiskit.synthesis import TwoQubitWeylDecomposition
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import Session, Batch

from qudits_on_qubits import create_ame_circuit, generate_b_ame
from sympy.functions.combinatorial.numbers import legendre_symbol
from IPython.display import display, Math
from itertools import product
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit.circuit import QuantumCircuit
from qiskit.circuit.library import StatePreparation
from igraph import Graph, plot
import matplotlib.pyplot as plt

In [ ]:
selected_candidate = "monomial_full__sup013_P102_ph100"
#QuditsOnQubits\artifacts\iqm_runs\selected_best\two_qutrit\stage2_top10_rerun20_20260706\exact\rank01_monomial_full__sup023_P012_ph022
artifact_circuit_dir = repo_root / "artifacts" / "iqm_runs" / "raw" / "quantum_circuits" / "garnet" / "two_qutrit" / selected_candidate
legacy_circuit_dir = repo_root.parent / "QuditsOnQubits" / "basis_direct_encoding_benchmarks" / "quantum_circuits" / "two_qutrit" / selected_candidate

for circuit_dir in (artifact_circuit_dir, legacy_circuit_dir):
    if (circuit_dir / "graph_state_direct_basis.qpy").is_file():
        break
else:
    raise FileNotFoundError(
        "Circuit artifacts not found. Checked:\n"
        f"- {artifact_circuit_dir}\n"
        f"- {legacy_circuit_dir}"
    )

print(f"Loading circuits from: {circuit_dir}")

with (circuit_dir / "graph_state_direct_basis.qpy").open("rb") as f:
    testqc = qpy.load(f)[0]

with (circuit_dir / "graph_state_direct_basis_transpiled.qpy").open("rb") as f:
    qcsuptrans = qpy.load(f)[0]

with (circuit_dir / "F3_W.qpy").open("rb") as f:
    F3sup = qpy.load(f)[0]

Esup = np.load(circuit_dir / "E.npy")

Loading circuits from: C:\Users\szymo\QuditsOnQubits\QuditsOnQubits\artifacts\iqm_runs\raw\quantum_circuits\garnet\two_qutrit\monomial_full__sup013_P102_ph022


In [51]:
from qudits_on_qubits.bell_measurements.sampler_circuits import build_sampler_circuits_from_graph, build_sampler_circuits_for_candidate
from iqm.qiskit_iqm.fake_backends.fake_garnet import IQMFakeGarnet
from iqm.iqm_client import CircuitCompilationOptions, DDMode, STANDARD_DD_STRATEGY, DDStrategy
from iqm.qiskit_iqm import IQMProvider
from iqm.qiskit_iqm.iqm_naive_move_pass import transpile_to_IQM
from iqm.iqm_client.transpile import ExistingMoveHandlingOptions

from qudits_on_qubits.bell_measurements.sampler_circuits import run_sampler_circuits_to_counts_by_setting
from qudits_on_qubits.bell_measurements.sampler_circuits import decoding_kwargs_from_metadata
from qudits_on_qubits.bell_measurements.postprocessing import compute_bell_value_from_counts

In [52]:
dd_options = CircuitCompilationOptions(
      dd_mode=DDMode.ENABLED,
      dd_strategy=STANDARD_DD_STRATEGY,
  )

dd_default = CircuitCompilationOptions(
      dd_mode=DDMode.ENABLED)

dd_xy4 = CircuitCompilationOptions(
      dd_mode=DDMode.ENABLED,
      dd_strategy=DDStrategy(
          gate_sequences=[(5, "YXYX", "asap")]
      ),
  )

In [53]:
import os
provider_garnet = IQMProvider("https://resonance.iqm.tech/", quantum_computer="garnet", token=os.environ["IQM_TOKEN"])
backend_garnet = provider_garnet.get_backend(use_metrics=True)

In [54]:
chosen_qubits = ["QB15", "QB16", "QB19", "QB20"]
chosen_indices = [backend_garnet.qubit_name_to_index(q) for q in chosen_qubits]

In [55]:
layout = qcsuptrans.layout.final_index_layout(filter_ancillas=True)
qutrit_qubits = ((layout[0], layout[1]), (layout[2], layout[3]))

sampler_circuits, metadata = build_sampler_circuits_for_candidate(candidate="two_qutrit", state_circuit=testqc, E=Esup, qutrit_qubits=((0, 1), (2, 3)))
isa_sampler_qc = [transpile_to_IQM(qc, backend=backend_garnet, optimization_level=3, seed_transpiler=3, initial_layout=chosen_indices) for qc in sampler_circuits]

In [56]:
isa_sampler_qc[0].depth()

26

In [57]:
counts_by_setting, run_info = run_sampler_circuits_to_counts_by_setting(isa_sampler_qc, metadata, shots=1024*20, transpile_circuits=False, backend=AerSampler())

In [ ]:
bell_value = compute_bell_value_from_counts(counts_by_setting, metadata["terms"], metadata["qutrit_bit_indices_by_setting"], **decoding_kwargs_from_metadata(metadata))
bell_value

(5.989591918051435-7.320533068622126e-15j)

In [ ]:
counts_by_setting_garnet, run_info_garnet = run_sampler_circuits_to_counts_by_setting(isa_sampler_qc, metadata, shots=1024*20, transpile_circuits=False, backend=backend_garnet, run_options={"circuit_compilation_options": dd_default})

APITimeoutError: The job 019f4203-a41b-75b2-a76c-f635b39861dd didn't finish in 10800.0 seconds.

In [ ]:
bell_value = compute_bell_value_from_counts(counts_by_setting_garnet, metadata["terms"], metadata["qutrit_bit_indices_by_setting"], **decoding_kwargs_from_metadata(metadata))
bell_value

(4.908125702660541-5.981326545168031e-15j)

Readout mitigation

In [ ]:
from provider import get_backend, get_backend_error_profile, generate_random_error_profile, to_static_architecture

In [ ]:
backend = get_backend(quantum_computer="garnet")
error_profile = generate_random_error_profile(backend=backend)

Backend connected successfully, using garnet

Noise profile: random-noise-profile

T1 / T2 / readout
qubit   T1 [us]   T2 [us]  readout 0->1  readout 1->0  readout avg error  readout asymmetry
  QB1 34.182868 24.987316      0.030764      0.022422           0.026593          -0.008342
  QB2 40.171518 21.353643      0.047997      0.052305           0.050151           0.004308
  QB3 49.688637 43.659762      0.028787      0.026290           0.027539          -0.002497
  QB4 25.131168 22.618052      0.050242      0.049116           0.049679          -0.001127
  QB5 52.977859 13.940976      0.015661      0.017561           0.016611           0.001899
  QB6 61.206720 54.256619      0.051943      0.060985           0.056464           0.009042
  QB7 39.678201 35.710381      0.034499      0.031625           0.033062          -0.002874
  QB8 70.357274 54.683926      0.018905      0.026286           0.022595           0.007380
  QB9 76.433874 24.762411      0.015678      0.007166           0.01142

In [ ]:
from iqm.qiskit_iqm.fake_backends.iqm_fake_backend import IQMFakeBackend
from iqm.qiskit_iqm.fake_backends.fake_garnet import IQMFakeGarnet
garnet = IQMFakeGarnet()
garnet_architecture = garnet.architecture

In [ ]:
static_garnet_architecture = to_static_architecture(garnet_architecture)

In [ ]:
garnet_noisy_backend = IQMFakeBackend(architecture=static_garnet_architecture, error_profile=error_profile)

In [ ]:
counts_by_setting_garnet_noise_model, run_info_garnet_noise_model = run_sampler_circuits_to_counts_by_setting(isa_sampler_qc, metadata, shots=1024*20, transpile_circuits=False, backend=garnet_noisy_backend)

In [ ]:
bell_value = compute_bell_value_from_counts(counts_by_setting_garnet_noise_model, metadata["terms"], metadata["qutrit_bit_indices_by_setting"], **decoding_kwargs_from_metadata(metadata))
bell_value

(3.1942287248725245-3.760880495917718e-15j)

In [ ]:
import mthree

In [ ]:
mapping = mthree.utils.final_measurement_mapping(isa_sampler_qc[0])

In [ ]:
mapping

{0: 19, 1: 15, 2: 18, 3: 14}

In [ ]:
measured_physical_qubits = sorted(set(mapping.values()))
measured_physical_qubits

[14, 15, 18, 19]

In [ ]:
from qiskit.circuit import QuantumCircuit
import numpy as np

def build_readout_calibration_matrices(
    backend,
    physical_qubits,
    shots=10_000
):
    """
    Zwraca listę macierzy kalibracyjnych M3.
    Dla kubitów, których nie kalibrujemy, pozostaje None.
    """
    matrices = [None] * backend.num_qubits

    for q in physical_qubits:
        # Przygotowanie |0> i pomiar kubitu q
        cal_0 = QuantumCircuit(backend.num_qubits, 1)
        cal_0.measure(q, 0)

        # Przygotowanie |1> i pomiar kubitu q
        cal_1 = QuantumCircuit(backend.num_qubits, 1)
        cal_1.x(q)
        cal_1.measure(q, 0)

        counts_0, counts_1 = backend.run(
            [cal_0, cal_1],
            shots=shots
        ).result().get_counts()

        # P(odczytano 1 | przygotowano 0)
        p10 = counts_0.get("1", 0) / shots

        # P(odczytano 0 | przygotowano 1)
        p01 = counts_1.get("0", 0) / shots

        # Kolumny: stan przygotowany |0>, |1>
        # Wiersze: stan odczytany  |0>, |1>
        matrices[q] = np.array(
            [
                [1 - p10, p01],
                [p10, 1 - p01]
            ],
            dtype=np.float32
        )

        print(f"Qubit {q}:")
        print(matrices[q])

    return matrices

In [ ]:
calibration_matrix = build_readout_calibration_matrices(backend_garnet, measured_physical_qubits, shots=10000)

Qubit 14:
[[0.9762 0.0104]
 [0.0238 0.9896]]
Qubit 15:
[[0.9857 0.0245]
 [0.0143 0.9755]]
Qubit 18:
[[0.979  0.0142]
 [0.021  0.9858]]
Qubit 19:
[[0.9816 0.0126]
 [0.0184 0.9874]]


In [ ]:
mit = mthree.M3Mitigation()

mit.cals_from_matrices(calibration_matrix)

In [ ]:
quasi = []

for res in list(counts_by_setting_garnet.values()):
    quasi.append(mit.apply_correction(res, mapping, return_mitigation_overhead=True))

In [ ]:
quasi_miti = {}

for i, setting in zip(quasi, list(counts_by_setting_garnet.keys())):
    quasi_temp = {}
    for key in i.keys():
        quasi_temp[key] = int(i[key]*1024*20)
    quasi_miti[setting] = quasi_temp

In [ ]:
bell_value = compute_bell_value_from_counts(quasi_miti, metadata["terms"], metadata["qutrit_bit_indices_by_setting"], **decoding_kwargs_from_metadata(metadata))
bell_value

(5.319001345741351-6.557254739192331e-15j)

In [ ]:
6 * math.cos(math.pi / 9)

5.638155724715451